# Demo 4：公共与私有 Skill

逐格调用接口，直接查看单元格返回值。使用 `.venv` 内核，操作当前 `config.yaml` 对应的技能目录。
更新接口后先重启内核，再初始化客户端。示例使用 `demo4-*` 技能；已有同名技能可跳过创建或换名。预设 ID 由创建接口随机生成，后续使用返回值中的 `preset_id`。


In [ ]:
from deerflow import DeerFlowSkillClient

skill_client = DeerFlowSkillClient(config_path="config.yaml")
public = skill_client.public


In [ ]:
skill_client

In [ ]:
public

## 一、公共 Skill

不需要 `user_id`。`list_skills()` 只返回技能信息；开关在 `get_preset()` 返回的预设详情中查询。

### 创建与查询


In [ ]:
public.list_skills()

In [ ]:
public.create_skill(
    name="demo4-summary",
    description="整理资料并生成摘要",
    tags="demo4, summary",
)


In [ ]:
public.create_skill(
    name="demo4-search",
    description="检索并整理资料来源",
    tags="demo4, search",
)


In [ ]:
public.get_skill_info(skill_name="demo4-summary")


In [ ]:
public.list_skills()

### 创建预设 1、2、3

创建时只传名称，返回随机 ID。此时未配置任何技能开关，后续单独调用 `update_preset()`。


In [ ]:
preset_1 = public.create_preset(name="预设 1：摘要")
preset_1


In [ ]:
preset_2 = public.create_preset(name="预设 2：检索")
preset_2


In [ ]:
preset_3 = public.create_preset(name="预设 3：综合")
preset_3


### 查询与修改预设

使用创建结果中的 `preset_id` 设置技能开关。


In [ ]:
public.update_preset(
    preset_id=preset_1["preset_id"],
    skill_states={"demo4-summary": True, "demo4-search": False},
)


In [ ]:
public.update_preset(
    preset_id=preset_2["preset_id"],
    skill_states={"demo4-summary": False, "demo4-search": True},
)


In [ ]:
public.list_presets()


In [ ]:
public.get_preset(preset_id=preset_1["preset_id"])


In [ ]:
public.update_preset(
    preset_id=preset_3["preset_id"],
    skill_states={"demo4-summary": False, "demo4-search": True},
)


In [ ]:
public.get_preset(preset_id=preset_3["preset_id"])

In [ ]:
public.list_skills()

### 默认预设与指定预设详情

保存原默认值，演示后恢复。设置默认值会立即写入文件 A；显式查询另一个预设不会修改默认值。


In [ ]:
original_default = public.get_preset()["preset_id"]
public.set_default_preset(preset_id=preset_1["preset_id"])

In [ ]:
original_default

In [ ]:
public.get_preset()


In [ ]:
public.list_skills()


In [ ]:
public.get_preset(preset_id=preset_2["preset_id"])


### 查看文件 A

可直接编辑此 JSON 文件。手改后重启内核，只重新运行初始化单元格和 `public.get_preset()` 即可查看加载结果。


In [ ]:
import json
import os
from pathlib import Path
from deerflow.config.paths import get_paths

preset_path = Path(os.environ.get("DEER_FLOW_SKILL_PRESETS_PATH") or get_paths().base_dir / "skill_presets.json")
preset_path


In [ ]:
json.loads(preset_path.read_text(encoding="utf-8-sig"))


### 恢复默认值与删除预设

当前默认预设不能直接删除；先恢复原默认值，再删除演示预设 3。预设 1、2 留作继续试用。


In [ ]:
public.set_default_preset(preset_id=original_default)


In [ ]:
public.delete_preset(preset_id=preset_3["preset_id"])


In [ ]:
public.list_presets()


### 文件与目录操作

先创建文件，再通过 `file_bytes` 上传 UTF-8 内容。


In [ ]:
public.create_folder(skill_name="demo4-summary", path="references")


In [ ]:
public.create_file(skill_name="demo4-summary", path="references/example.txt")


In [ ]:
public.upload_file(
    skill_name="demo4-summary",
    path="references/example.txt",
    file_bytes="公共摘要示例".encode("utf-8"),
)


In [ ]:
public.download_file(skill_name="demo4-summary", path="references/example.txt").decode("utf-8")


In [ ]:
public.move_file(
    skill_name="demo4-summary",
    path="references/example.txt",
    new_path="references/renamed.txt",
)


In [ ]:
public.move_folder(skill_name="demo4-summary", path="references", new_path="examples")


In [ ]:
public.get_resource_tree(skill_name="demo4-summary")


In [ ]:
public.delete_file(skill_name="demo4-summary", path="examples/renamed.txt")


In [ ]:
public.delete_folder(skill_name="demo4-summary", path="examples")


### 删除公共技能包


In [ ]:
public.create_skill(name="demo4-temporary", description="用于演示删除")


In [ ]:
public.delete_skill(skill_name="demo4-temporary")

### stream 选择预设

以下为调用示例，`pool` 使用服务中已配置模型和沙箱的 `DeerFlowClientPool`，与管理接口使用同一套技能目录和文件 A。

```python
with pool.session(user_id="demo4_alice") as client:
    with client.stream("整理资料", skill_preset=preset_1["preset_id"]) as events:
        result = list(events)
result
```

省略 `skill_preset`，本轮使用文件 A 中的默认预设：

```python
with pool.start_stream("整理资料", user_id="demo4_alice") as events:
    result = list(events)
result
```


## 二、私有 Skill

必须传 `user_id`，没有预设。`get_skill_info()` 额外返回最终 `enabled` 状态。

### 创建与查询


In [ ]:
skill_client.create_skill(
    user_id="demo4_alice",
    name="demo4-summary",
    description="Alice 的私有摘要习惯",
    tags="demo4, private",
)


In [ ]:
skill_client.get_skill_info(user_id="demo4_alice", skill_name="demo4-summary")


In [ ]:
skill_client.list_skills(user_id="demo4_alice")


### 私有开关

只修改当前用户的私有状态，不修改公共预设。私有最终开关仍是 `global_enabled AND user_enabled`。


In [ ]:
skill_client.list_skill_states(user_id="demo4_alice")


In [ ]:
skill_client.set_skill_enabled(user_id="demo4_alice", skill_name="demo4-summary", enabled=False)


In [ ]:
skill_client.get_skill_info(user_id="demo4_alice", skill_name="demo4-summary")


In [ ]:
skill_client.set_skill_enabled(user_id="demo4_alice", skill_name="demo4-summary", enabled=True)


In [ ]:
skill_client.get_skill_info(user_id="demo4_alice", skill_name="demo4-summary")


### 用户之间复制


In [ ]:
skill_client.copy_skill(
    source_user_id="demo4_alice",
    target_user_id="demo4_bob",
    skill_name="demo4-summary",
)


In [ ]:
skill_client.list_skills(user_id="demo4_bob")


### 公共复制到私有


In [ ]:
public.copy_skill(skill_name="demo4-search", target_user_id="demo4_alice")


In [ ]:
skill_client.list_skills(user_id="demo4_alice")


### 文件与目录操作


In [ ]:
skill_client.create_folder(user_id="demo4_alice", skill_name="demo4-summary", path="references")


In [ ]:
skill_client.create_file(user_id="demo4_alice", skill_name="demo4-summary", path="references/notes.txt")


In [ ]:
skill_client.upload_file(
    user_id="demo4_alice",
    skill_name="demo4-summary",
    path="references/notes.txt",
    file_bytes="Alice 的私有笔记".encode("utf-8"),
)


In [ ]:
skill_client.download_file(user_id="demo4_alice", skill_name="demo4-summary", path="references/notes.txt").decode("utf-8")


In [ ]:
skill_client.move_file(
    user_id="demo4_alice",
    skill_name="demo4-summary",
    path="references/notes.txt",
    new_path="references/renamed.txt",
)


In [ ]:
skill_client.move_folder(
    user_id="demo4_alice",
    skill_name="demo4-summary",
    path="references",
    new_path="notes",
)


In [ ]:
skill_client.get_resource_tree(user_id="demo4_alice", skill_name="demo4-summary")


### 更新 SKILL.md

通过完整文件上传更新，`name` 必须与技能名称一致。


In [ ]:
skill_client.upload_file(
    user_id="demo4_alice",
    skill_name="demo4-summary",
    path="SKILL.md",
    file_bytes="---\nname: demo4-summary\ndescription: Alice 更新后的摘要习惯\n---\n\n先列结论，再列依据。\n".encode("utf-8"),
)


In [ ]:
skill_client.download_file(user_id="demo4_alice", skill_name="demo4-summary", path="SKILL.md").decode("utf-8")


### 删除文件、目录与技能包

删除演示文件和复制的技能，保留 Alice 的摘要技能。


In [ ]:
skill_client.delete_file(user_id="demo4_alice", skill_name="demo4-summary", path="notes/renamed.txt")


In [ ]:
skill_client.delete_folder(user_id="demo4_alice", skill_name="demo4-summary", path="notes")


In [ ]:
skill_client.delete_skill(user_id="demo4_alice", skill_name="demo4-search")


In [ ]:
skill_client.delete_skill(user_id="demo4_bob", skill_name="demo4-summary")


In [ ]:
skill_client.list_skills(user_id="demo4_alice")


In [ ]:
skill_client.list_skills(user_id="demo4_alice")
